In [67]:
import os
import pandas as pd
from dotenv import load_dotenv
from gsheet_loader import load_sheet

load_dotenv()

# Load a Google Sheet URL from .env
url = os.getenv("GOOGLE_SHEET_URL")
if not url:
    raise ValueError("GOOGLE_SHEET_URL is not set in .env")

data = load_sheet(url)

In [68]:
df = pd.read_csv(data)

In [69]:
df.drop('submittedAt', axis=1, inplace=True)

In [70]:
df['B5_time_with_parents_hours'] = df['B5_time_with_parents_hours'].str.replace('â', '-', regex=False)
df['D1_daily_screen'] = df['D1_daily_screen'].str.replace('â', '-', regex=False)
df['D2_weekend_screen'] = df['D2_weekend_screen'].str.replace('â', '-', regex=False)
df['D5_parents_screen_time'] = df['D5_parents_screen_time'].str.replace('â', '-', regex=False)
df['E1_sleep_duration'] = df['E1_sleep_duration'].str.replace('â', '-', regex=False)

## **Outlier Detection**

In [71]:
# Inspect raw values to diagnose parsing issue
print("Unique values in B5_time_with_parents_hours:")
print(df['B5_time_with_parents_hours'].unique())

print("\nSample repr (shows hidden characters):")
for v in df['B5_time_with_parents_hours'].dropna().unique()[:10]:
    print(repr(v))


Unique values in B5_time_with_parents_hours:
['2-4 hours' '4-6 hours' '1-2 hours' 'More than 6 hours'
 'Less than 1 hour']

Sample repr (shows hidden characters):
'2-4 hours'
'4-6 hours'
'1-2 hours'
'More than 6 hours'
'Less than 1 hour'


In [72]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

all_cols = list(df.columns)
numeric_cols = []
categorical_cols = []

for col in all_cols:
    converted = pd.to_numeric(df[col], errors='coerce')
    if converted.notna().sum() >= len(df) * 0.5:
        numeric_cols.append(col)
    else:
        categorical_cols.append(col)

# ── Compute summaries ────────────────────────────────────────────────────────
num_summary = []
for col in numeric_cols:
    series = pd.to_numeric(df[col], errors='coerce').dropna()
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    iqr_out = series[(series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)]
    z_scores = (series - series.mean()) / series.std()
    z_out = series[z_scores.abs() > 3]
    num_summary.append({
        'Column': col, 'N': len(series),
        'Mean': round(series.mean(), 2), 'Std': round(series.std(), 2),
        'IQR Outliers': len(iqr_out), 'Z-Score Outliers': len(z_out)
    })

cat_summary = []
for col in categorical_cols:
    counts = df[col].value_counts(dropna=True)
    freq = counts.values.astype(float)
    Q1, Q3 = np.percentile(freq, 25), np.percentile(freq, 75)
    IQR = Q3 - Q1
    iqr_mask = (freq < Q1 - 1.5 * IQR) | (freq > Q3 + 1.5 * IQR)
    iqr_out_cats = counts.index[iqr_mask].tolist()
    mean, std = freq.mean(), freq.std()
    z_sc = (freq - mean) / std if std > 0 else np.zeros(len(freq))
    z_mask = np.abs(z_sc) > 3
    z_out_cats = counts.index[z_mask].tolist()
    cat_summary.append({
        'Column': col,
        'Unique': len(counts),
        'Mean Freq': round(mean, 1),
        'Std Freq': round(std, 1),
        'IQR Outlier\nCategories': len(iqr_out_cats),
        'Z-Score Outlier\nCategories': len(z_out_cats),
        'IQR Outlier Values': (', '.join(str(v) for v in iqr_out_cats[:2]) + ('...' if len(iqr_out_cats) > 2 else '')) or 'None',
        'Z-Score Outlier Values': (', '.join(str(v) for v in z_out_cats[:2]) + ('...' if len(z_out_cats) > 2 else '')) or 'None'
    })

num_df = pd.DataFrame(num_summary)
cat_df = pd.DataFrame(cat_summary)


def title_page(pdf, title, subtitle, stats_text):
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.patch.set_facecolor('white')
    ax = fig.add_subplot(111)
    ax.axis('off')
    ax.text(0.5, 0.65, title, ha='center', va='center',
            fontsize=22, fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.57, subtitle, ha='center', va='center',
            fontsize=13, transform=ax.transAxes, color='#555555')
    ax.text(0.5, 0.50, stats_text, ha='center', va='center',
            fontsize=10, transform=ax.transAxes, color='#777777')
    pdf.savefig(fig, facecolor='white')
    plt.close()


def summary_table_page(pdf, dataframe, title):
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.patch.set_facecolor('white')
    ax = fig.add_subplot(111)
    ax.axis('off')
    ax.text(0.5, 0.97, title, ha='center', va='top',
            fontsize=12, fontweight='bold', transform=ax.transAxes)

    col_widths = [max(len(str(v)) for v in [c] + dataframe[c].tolist()) for c in dataframe.columns]
    total = sum(col_widths)
    col_widths = [w / total for w in col_widths]

    tbl = ax.table(
        cellText=dataframe.values,
        colLabels=dataframe.columns,
        cellLoc='center', loc='center',
        colWidths=col_widths,
        bbox=[0.0, 0.02, 1.0, 0.92]
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor('#dddddd')
        cell.set_linewidth(0.5)
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold', fontsize=8)
        else:
            cell.set_facecolor('#f2f2f2' if row % 2 == 0 else 'white')
            cell.set_text_props(fontsize=8)
    pdf.savefig(fig, facecolor='white')
    plt.close()


# ════════════════════════════════════════════════════════════════════════
# PDF 1 — NUMERIC
# ════════════════════════════════════════════════════════════════════════
with PdfPages('outlier_numeric.pdf') as pdf:

    title_page(pdf,
               'Outlier Detection Report',
               'Numeric Columns — IQR & Z-Score Analysis',
               f'Total Rows: {len(df)}   |   Numeric Columns: {len(numeric_cols)}')

    summary_table_page(pdf, num_df, 'Numeric Columns — Outlier Summary (IQR + Z-Score)')

    # Boxplots — 2 rows x 2 cols per page
    per_page = 4
    for start in range(0, len(numeric_cols), per_page):
        batch = numeric_cols[start:start + per_page]
        n = len(batch)
        cols_r = 2
        rows_r = (n + cols_r - 1) // cols_r
        fig, axes = plt.subplots(rows_r, cols_r, figsize=(8.27, 11.69))
        fig.patch.set_facecolor('white')
        fig.suptitle('Numeric Columns — IQR Outliers', fontsize=13, fontweight='bold', y=0.98)
        axes = np.array(axes).flatten()
        for i, col in enumerate(batch):
            series = pd.to_numeric(df[col], errors='coerce').dropna()
            Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
            IQR = Q3 - Q1
            iqr_out = series[(series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)]
            axes[i].boxplot(series, vert=True, patch_artist=True,
                            boxprops=dict(facecolor='#d6eaf8', color='#2c3e50'),
                            medianprops=dict(color='#e74c3c', linewidth=2))
            if len(iqr_out):
                axes[i].scatter([1]*len(iqr_out), iqr_out, color='red',
                                zorder=5, s=30, label=f'{len(iqr_out)} outlier(s)')
                axes[i].legend(fontsize=8, loc='upper right')
            axes[i].set_title(col, fontsize=10, fontweight='bold', pad=6)
            axes[i].tick_params(labelsize=8)
            axes[i].set_ylabel('Value', fontsize=8)
            # Annotation
            info = (f'Q1={Q1:.1f}  Q3={Q3:.1f}  IQR={IQR:.1f}\n'
                    f'Lower={Q1-1.5*IQR:.1f}  Upper={Q3+1.5*IQR:.1f}')
            axes[i].annotate(info, xy=(0.02, 0.98), xycoords='axes fraction',
                             fontsize=7, va='top', color='#555555',
                             bbox=dict(boxstyle='round,pad=0.3', facecolor='#fdfefe', alpha=0.7))
        for j in range(len(batch), len(axes)):
            axes[j].set_visible(False)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        pdf.savefig(fig, facecolor='white')
        plt.close()

print("Saved: outlier_numeric.pdf")


# ════════════════════════════════════════════════════════════════════════
# PDF 2 — CATEGORICAL
# ════════════════════════════════════════════════════════════════════════
with PdfPages('outlier_categorical.pdf') as pdf:

    title_page(pdf,
               'Outlier Detection Report',
               'Categorical Columns — IQR & Z-Score on Category Frequencies',
               f'Total Rows: {len(df)}   |   Categorical Columns: {len(categorical_cols)}')

    summary_table_page(pdf, cat_df, 'Categorical Columns — Outlier Summary (IQR + Z-Score on Frequencies)')

    # Bar charts — 2 per page
    per_page = 2
    for start in range(0, len(categorical_cols), per_page):
        batch = categorical_cols[start:start + per_page]
        fig, axes = plt.subplots(1, per_page, figsize=(8.27, 11.69))
        fig.patch.set_facecolor('white')
        fig.suptitle('Categorical Columns — IQR Outlier Categories in Red',
                     fontsize=13, fontweight='bold', y=0.99)
        axes = np.array(axes).flatten()
        for i, col in enumerate(batch):
            counts = df[col].value_counts(dropna=True)
            freq = counts.values.astype(float)
            Q1, Q3 = np.percentile(freq, 25), np.percentile(freq, 75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            iqr_mask = (freq < lower) | (freq > upper)
            colors = ['#e74c3c' if m else '#2980b9' for m in iqr_mask]

            bars = axes[i].bar(range(len(counts)), freq, color=colors, edgecolor='white', linewidth=0.5)
            axes[i].set_xticks(range(len(counts)))
            axes[i].set_xticklabels(
                [lbl[:25] + '…' if len(str(lbl)) > 25 else str(lbl) for lbl in counts.index],
                rotation=40, ha='right', fontsize=7
            )
            axes[i].axhline(y=upper, color='orange', linestyle='--', linewidth=1.2, label=f'Upper={upper:.1f}')
            if lower > 0:
                axes[i].axhline(y=lower, color='orange', linestyle=':', linewidth=1.2, label=f'Lower={lower:.1f}')
            axes[i].set_title(col, fontsize=10, fontweight='bold', pad=6)
            axes[i].set_ylabel('Frequency', fontsize=8)
            axes[i].tick_params(axis='y', labelsize=8)

            # Value labels on bars
            for bar, val in zip(bars, freq):
                axes[i].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                             str(int(val)), ha='center', va='bottom', fontsize=7)

            axes[i].legend(fontsize=7, loc='upper right')

            n_out = int(iqr_mask.sum())
            axes[i].annotate(f'IQR Outlier Categories: {n_out}',
                             xy=(0.02, 0.98), xycoords='axes fraction',
                             fontsize=8, va='top', color='#c0392b',
                             bbox=dict(boxstyle='round,pad=0.3', facecolor='#fdfefe', alpha=0.8))

        for j in range(len(batch), per_page):
            axes[j].set_visible(False)
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        pdf.savefig(fig, facecolor='white')
        plt.close()

print("Saved: outlier_categorical.pdf")


Saved: outlier_numeric.pdf
Saved: outlier_categorical.pdf
